In [1]:
import pandas as pd
import requests
import time
import os

# ==========================================
# LOAD COORDINATES
# ==========================================

coords_df = pd.read_csv(
    "../datasets/weather/multi_state_coordinates.csv"
)

print(coords_df.head())

# ==========================================
# STORE RESULTS
# ==========================================

weather_data = []

# ==========================================
# LOOP THROUGH DISTRICTS
# ==========================================

for index, row in coords_df.iterrows():

    district = row['District_Name']
    state = row['State_Name']
    lat = row['Latitude']
    lon = row['Longitude']

    print(f"\nProcessing: {district} ({state})")

    # NASA POWER API URL (Daily endpoint for 10-year span in 1 request)
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=T2M,RH2M,PRECTOTCORR"
        f"&community=AG"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start=20150101"
        f"&end=20241231"
        f"&format=JSON"
    )

    success = False
    for attempt in range(3):
        try:
            response = requests.get(url, timeout=20)
            if response.status_code == 200:
                data_resp = response.json()
                params = data_resp['properties']['parameter']

                # Aggregate daily observations to yearly averages
                yearly_data = {}
                for date_str in params['T2M']:
                    year = int(date_str[:4])
                    if year not in yearly_data:
                        yearly_data[year] = {'T2M': [], 'RH2M': [], 'PRECTOTCORR': []}
                    yearly_data[year]['T2M'].append(params['T2M'][date_str])
                    yearly_data[year]['RH2M'].append(params['RH2M'][date_str])
                    yearly_data[year]['PRECTOTCORR'].append(params['PRECTOTCORR'][date_str])

                for year in sorted(yearly_data.keys()):
                    avg_t2m = sum(yearly_data[year]['T2M']) / len(yearly_data[year]['T2M'])
                    avg_rh2m = sum(yearly_data[year]['RH2M']) / len(yearly_data[year]['RH2M'])
                    total_rain = sum(yearly_data[year]['PRECTOTCORR'])

                    weather_data.append({
                        'State_Name': state.strip().lower(),
                        'District_Name': district.strip().lower(),
                        'Year': year,
                        'T2M': avg_t2m,
                        'RH2M': avg_rh2m,
                        'PRECTOTCORR': total_rain
                    })
                print("  2015-2024 DONE")
                success = True
                break
            else:
                print(f"  Attempt {attempt+1} failed with status code {response.status_code}")
        except Exception as e:
            print(f"  Attempt {attempt+1} encountered error: {e}")
        time.sleep(2)

    if not success:
        print(f"ERROR: {district}")

    # Compliance with NASA rate limit rules
    time.sleep(0.5)

# ==========================================
# CREATE DATAFRAME
# ==========================================

weather_df = pd.DataFrame(weather_data)

print("\nFINAL WEATHER DATASET:")
print(weather_df.head())

print("\nTOTAL ROWS:")
print(len(weather_df))

# ==========================================
# SAVE
# ==========================================

weather_df.to_csv(
    "../datasets/weather/multi_state_weather.csv",
    index=False
)

print("\nMULTI-STATE WEATHER DATASET SAVED!")


      State_Name   District_Name  Latitude  Longitude
0  Uttar Pradesh            Agra   27.1767    78.0081
1      Rajasthan           Ajmer   26.4500    74.6400
2  Uttar Pradesh         Aligarh   27.8974    78.0880
3      Rajasthan           Alwar   27.5667    76.6333
4  Uttar Pradesh  Ambedkar Nagar   26.4050    82.8390

Processing: Agra (Uttar Pradesh)


  2015-2024 DONE



Processing: Ajmer (Rajasthan)


  2015-2024 DONE



Processing: Aligarh (Uttar Pradesh)


...


Processing: Unnao (Uttar Pradesh)


  2015-2024 DONE



Processing: Varanasi (Uttar Pradesh)


  2015-2024 DONE



FINAL WEATHER DATASET:
      State_Name District_Name  Year        T2M       RH2M  PRECTOTCORR
0  uttar pradesh          agra  2015  26.391288  43.295863       589.35
1  uttar pradesh          agra  2016  26.953716  42.146858       771.76
2  uttar pradesh          agra  2017  27.219096  39.360767       454.06
3  uttar pradesh          agra  2018  26.341507  46.327671       857.98
4  uttar pradesh          agra  2019  26.030219  50.758603       698.90

TOTAL ROWS:
1080

MULTI-STATE WEATHER DATASET SAVED!
